# Configure and run your first hsB-SM model

Install the package and prepare your catchment inputs using the repository
[Quick Start](../README.md#quick-start). Copy `templates/catchment` to `my_model`,
fill the geometry, channel, and forcing tables, and review `config.json`.
This notebook reads your completed inputs. It supplies no meteorological data,
catchment geometry, or simulated example results.

Install notebook support with `python -m pip install -e ".[notebook]"` from the
repository root, then start `jupyter lab`.


In [ ]:
from pathlib import Path
from hsbsm import load_config, run

root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "pyproject.toml").is_file() and (p / "templates/catchment").is_dir()),
    Path.cwd(),
)
config_path = root / "my_model/config.json"  # Change this path for your project.
output_path = root / "outputs/my_model"
print("Configuration:", config_path)


## Check the configuration

Supply positive cell widths and lengths, an increasing channel-distance
distribution, and a meteorological table with equally spaced timestamps.
Use the column names and units in the Quick Start. Set `dt` in seconds,
`channel.slope` in m/m, and `parameters.Hill.iota_rad` in radians. These three
required fields are initially null in the template.

Choose the LAI area basis and interception setting, then review soil, aquifer,
vegetation, and routing parameters and initial stores. The remaining parameter
defaults are not calibrated estimates for your catchment.


In [ ]:
import json

cfg = None
if not config_path.is_file():
    print("Prepare your input files and set config_path before continuing.")
else:
    spec = json.loads(config_path.read_text(encoding="utf-8"))
    required = {
        "dt": spec.get("dt"),
        "channel.slope": spec.get("channel", {}).get("slope"),
        "parameters.Hill.iota_rad": spec.get("parameters", {}).get("Hill", {}).get("iota_rad"),
    }
    missing = [name for name, value in required.items() if value is None]
    if missing:
        print("Complete these required settings:", ", ".join(missing))
    else:
        cfg = load_config(config_path)
        print(len(cfg.forcing), "forcing intervals;", len(cfg.grid.x), "hillslope cells")
        print("Timestep (s):", cfg.dt)
        print("Hillslope area (m²):", cfg.grid.area.sum())


## Run with the prepared inputs

Run the following cell after the configuration loads successfully and you have
reviewed the parameter values. Choose a new output directory for each run;
saving to an existing directory replaces files with the same names.


In [ ]:
result = None
if cfg is None:
    print("Complete the input preparation and configuration check first.")
else:
    result = run(cfg)
    result.save(output_path)
    print("Results saved to:", output_path)


## Inspect the results

Check the water-balance residual, dates, units, discharge, and soil water
contents. The exported figures describe this configured run. Water conservation
is a numerical check; evaluation of predictive performance requires observed
data. CSV tables, node arrays, parameters, and diagnostic files are saved with
the figures.


In [ ]:
if result is not None:
    from IPython.display import Image, display
    print("Relative water-balance error:", result.diagnostics["relative_water_balance_error"])
    display(result.timeseries[["date", "Q_total_m3s", "ET_mm_day"]].head())
    display(Image(filename=str(output_path / "hydrograph.png")))
